In [10]:
!pip install pyspark -q

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Part 1: Clustering (Farthest-First & k-Means++)
Implementation of the Farthest First traversal for k-center clustering and the k-means++ initialization algorithm.

In [12]:
import numpy as np
import random
import time
import os

def readVectorsSeq(filename):
    """Reads feature vectors from a text file into a list of numpy arrays."""
    data_points = []
    with open(filename, "r") as file:
        for line in file:
            line = line.strip()
            if line:
                features = [float(val) for val in line.split(",")]
                data_points.append(np.array(features, dtype=np.float64))
    return data_points

def kcenter(P, k):
    """Farthest-First Traversal for k-center clustering."""
    num_points = len(P)
    chosen_centers = []
    # Track the shortest distance from each point to any chosen center
    closest_dists = np.full(num_points, np.inf)

    # Pick the first center randomly
    first_idx = random.randint(0, num_points - 1)
    chosen_centers.append(P[first_idx])

    for _ in range(k - 1):
        latest_center = chosen_centers[-1]

        # Update distances based on the newly added center
        for i in range(num_points):
            sq_dist = float(np.sum((P[i] - latest_center) ** 2))
            if sq_dist < closest_dists[i]:
                closest_dists[i] = sq_dist

        # The next center is the point with the maximum minimum distance
        farthest_idx = int(np.argmax(closest_dists))
        chosen_centers.append(P[farthest_idx])

    return chosen_centers

def kmeansPP(P, k):
    """k-Means++ initialization algorithm."""
    num_points = len(P)
    chosen_centers = []
    closest_dists = np.full(num_points, np.inf)

    # First center is random
    first_idx = random.randint(0, num_points - 1)
    chosen_centers.append(P[first_idx])

    for _ in range(k - 1):
        latest_center = chosen_centers[-1]

        for i in range(num_points):
            sq_dist = float(np.sum((P[i] - latest_center) ** 2))
            if sq_dist < closest_dists[i]:
                closest_dists[i] = sq_dist

        sum_dists = closest_dists.sum()
        if sum_dists == 0:
            next_idx = random.randint(0, num_points - 1)
        else:
            # Probability proportional to squared distance
            probabilities = closest_dists / sum_dists
            next_idx = int(np.random.choice(num_points, p=probabilities))

        chosen_centers.append(P[next_idx])

    return chosen_centers

def kmeansObj(P, C):
    """Calculates the average squared distance from points to their closest center."""
    centers_matrix = np.array(C)
    total_cost = 0.0

    for point in P:
        # Calculate distances to all centers and find the minimum
        distances = np.sum((centers_matrix - point) ** 2, axis=1)
        total_cost += float(np.min(distances))

    return total_cost / len(P)

In [13]:

BASE_PATH = r"/content/drive/MyDrive/Datasets/MLBD/Assignment4/"
# --- Running the Part 1 Experiment ---
if __name__ == "__main__":
    spam_data_file = os.path.join(BASE_PATH, "Q1- UCI Spam clustering", "spambase.data")

    print("Loading dataset...")
    P = readVectorsSeq(spam_data_file)
    print(f"Loaded {len(P)} points with {len(P[0])} features each.\n")

    k = 10
    k1 = 50

    # Task 1: k-center time
    start_time = time.time()
    C_kcenter = kcenter(P, k)
    print(f"[Task 1] kcenter (k={k}) execution time: {time.time() - start_time:.4f} seconds")

    # Task 2: kmeans++ objective
    C_kmeanspp = kmeansPP(P, k)
    obj_kmeanspp = kmeansObj(P, C_kmeanspp)
    print(f"[Task 2] kmeans++ (k={k}) objective score: {obj_kmeanspp:.4f}")

    # Task 3: kcenter coreset -> kmeans++
    X_coreset = kcenter(P, k1)
    C_coreset_kmeans = kmeansPP(X_coreset, k)
    obj_coreset = kmeansObj(P, C_coreset_kmeans)
    print(f"[Task 3] kcenter (k1={k1}) -> kmeans++ (k={k}) objective score: {obj_coreset:.4f}")

Loading dataset...
Loaded 4601 points with 58 features each.

[Task 1] kcenter (k=10) execution time: 1.7711 seconds
[Task 2] kmeans++ (k=10) objective score: 24081.9900
[Task 3] kcenter (k1=50) -> kmeans++ (k=10) objective score: 194762.4100


## Part 2: Web-Search
Building an inverted index to answer basic search queries using custom data structures.

In [14]:
import os
import math

WEBPAGES_DIR = os.path.join(BASE_PATH, "Q2- webSearch", "webpages")

STOP_WORDS = {"a", "an", "the", "they", "these", "this", "for", "is", "are",
              "was", "of", "or", "and", "does", "will", "whose"}
PUNCTUATION = set("{}[]<>=().,;'\"?#!-:")

PLURAL_TO_SINGULAR = {
    "stacks": "stack",
    "structures": "structure",
    "applications": "application",
}

def clean_and_normalize(word):
    word = word.lower()
    return PLURAL_TO_SINGULAR.get(word, word)

def extract_tokens(text):
    """Yields (position_index, valid_word) from text document."""
    tokens = text.split()
    pos_counter = 0
    valid_tokens = []

    for token in tokens:
        # Strip punctuation
        clean_word = "".join([char for char in token if char not in PUNCTUATION])
        if not clean_word:
            continue

        pos_counter += 1
        lowered = clean_word.lower()

        # Skip storing if it's a stop word, but keep position count
        if lowered in STOP_WORDS:
            continue

        normalized = PLURAL_TO_SINGULAR.get(lowered, lowered)
        valid_tokens.append((pos_counter, normalized))

    return valid_tokens

class MySet:
    def __init__(self):
        self.elements = set()

    def addElement(self, element):
        self.elements.add(element)

    def union(self, otherSet):
        new_set = MySet()
        new_set.elements = self.elements.union(otherSet.elements)
        return new_set

    def intersection(self, otherSet):
        new_set = MySet()
        new_set.elements = self.elements.intersection(otherSet.elements)
        return new_set

    def __len__(self):
        return len(self.elements)

    def __iter__(self):
        return iter(self.elements)

class Position:
    def __init__(self, p, wordIndex):
        self.page = p
        self.index = wordIndex

    def getPageEntry(self):
        return self.page

    def getWordIndex(self):
        return self.index

class WordEntry:
    def __init__(self, word):
        self.word_str = word
        self.positions_list = []

    def addPosition(self, position):
        self.positions_list.append(position)

    def addPositions(self, positions):
        self.positions_list.extend(positions)

    def getAllPositionsForThisWord(self):
        return self.positions_list

    def getWord(self):
        return self.word_str

    def getTermFrequency(self, pageEntry):
        occurrences = sum(1 for p in self.positions_list if p.getPageEntry() == pageEntry)
        total_page_words = pageEntry.getTotalWords()
        return occurrences / total_page_words if total_page_words > 0 else 0

    def getPagesContaining(self):
        pages_set = MySet()
        for pos in self.positions_list:
            pages_set.addElement(pos.getPageEntry())
        return pages_set

class PageIndex:
    def __init__(self):
        self.word_dict = {}

    def addPositionForWord(self, string, p):
        if string not in self.word_dict:
            self.word_dict[string] = WordEntry(string)
        self.word_dict[string].addPosition(p)

    def getWordEntries(self):
        return list(self.word_dict.values())

    def getEntryForWord(self, w):
        return self.word_dict.get(w, None)

class PageEntry:
    def __init__(self, pageName):
        self.name = pageName
        self.p_index = PageIndex()
        self.total_word_count = 0

        file_path = os.path.join(WEBPAGES_DIR, pageName)
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()

        self.total_word_count = len(content.split())

        for pos_idx, word in extract_tokens(content):
            pos_obj = Position(self, pos_idx)
            self.p_index.addPositionForWord(word, pos_obj)

    def getPageIndex(self):
        return self.p_index

    def getPageName(self):
        return self.name

    def getTotalWords(self):
        return self.total_word_count

class MyHashTable:
    def __init__(self):
        self.capacity = 1009
        self.buckets = [[] for _ in range(self.capacity)]

    def getHashIndex(self, string):
        hash_val = 0
        for char in string:
            hash_val = (hash_val * 31 + ord(char)) % self.capacity
        return hash_val

    def addPositionsForWord(self, w):
        word_str = w.getWord()
        bucket_idx = self.getHashIndex(word_str)

        for existing_entry in self.buckets[bucket_idx]:
            if existing_entry.getWord() == word_str:
                existing_entry.addPositions(w.getAllPositionsForThisWord())
                return

        new_entry = WordEntry(word_str)
        new_entry.addPositions(w.getAllPositionsForThisWord())
        self.buckets[bucket_idx].append(new_entry)

    def fetchWordEntry(self, string):
        bucket_idx = self.getHashIndex(string)
        for entry in self.buckets[bucket_idx]:
            if entry.getWord() == string:
                return entry
        return None

class InvertedPageIndex:
    def __init__(self):
        self.hash_table = MyHashTable()
        self.registered_pages = {}

    def addPage(self, p):
        self.registered_pages[p.getPageName()] = p
        for w_entry in p.getPageIndex().getWordEntries():
            self.hash_table.addPositionsForWord(w_entry)

    def getPagesWhichContainWord(self, string):
        w_entry = self.hash_table.fetchWordEntry(string)
        if not w_entry:
            return MySet()
        return w_entry.getPagesContaining()

class SearchEngine:
    def __init__(self):
        self.inv_index = InvertedPageIndex()

    def performAction(self, actionMessage):
        tokens = actionMessage.strip().split()
        if not tokens:
            return

        command = tokens[0]

        if command == "addPage":
            p_name = tokens[1]
            p_entry = PageEntry(p_name)
            self.inv_index.addPage(p_entry)

        elif command == "queryFindPagesWhichContainWord":
            target_word = tokens[1]
            norm_word = clean_and_normalize(target_word)
            found_pages = self.inv_index.getPagesWhichContainWord(norm_word)

            if len(found_pages) == 0:
                print(f"No webpage contains word {target_word}")
            else:
                sorted_names = sorted(p.getPageName() for p in found_pages)
                print(", ".join(sorted_names))

        elif command == "queryFindPositionsOfWordInAPage":
            target_word = tokens[1]
            doc_name = tokens[2]
            norm_word = clean_and_normalize(target_word)

            if doc_name not in self.inv_index.registered_pages:
                print(f"No webpage {doc_name} found")
                return

            page_obj = self.inv_index.registered_pages[doc_name]
            word_obj = page_obj.getPageIndex().getEntryForWord(norm_word)

            if word_obj is None:
                print(f"Webpage {doc_name} does not contain word {target_word}")
            else:
                positions = sorted(p.getWordIndex() for p in word_obj.getAllPositionsForThisWord())
                print(", ".join(map(str, positions)))


In [15]:
WEBPAGES_DIR = "/content/drive/MyDrive/Datasets/MLBD/Assignment4/websearch"
# --- Run Search Engine Actions ---
if __name__ == "__main__":
    engine = SearchEngine()
    actions_path = os.path.join(WEBPAGES_DIR, "actions.txt")

    print("Executing Search Queries...\n")
    with open(actions_path, "r") as f:
        for line in f:
            if line.strip():
                engine.performAction(line.strip())

Executing Search Queries...

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


## Part 3: PageRank on Spark
Implementing the iterative PageRank algorithm using PySpark RDDs.

In [16]:
def run_pagerank(file_path, num_nodes, beta=0.8, num_iters=40):
    # 1. Load data and remove duplicate edges
    raw_edges = sc.textFile(file_path)
    edges = raw_edges.map(lambda line: tuple(map(int, line.strip().split()))).distinct().cache()

    # 2. Calculate out-degree for each node
    out_degrees = edges.map(lambda e: (e[0], 1)).reduceByKey(lambda x, y: x + y).cache()

    # 3. Create adjacency graph and PARTITION it to prevent shuffling in the loop
    # We use 4 partitions for a small graph, but you can adjust this.
    adj_graph = edges.join(out_degrees)\
                     .map(lambda val: (val[0], (val[1][0], 1.0 / val[1][1])))\
                     .partitionBy(4)\
                     .cache()

    # 4. Initialize rank vector and PARTITION it
    ranks = sc.parallelize(range(1, num_nodes + 1))\
              .map(lambda node: (node, 1.0 / num_nodes))\
              .partitionBy(4)\
              .cache()

    teleport_val = (1.0 - beta) / num_nodes

    # Pre-compute the base nodes to avoid doing this 40 times
    all_nodes_base = sc.parallelize(range(1, num_nodes + 1))\
                       .map(lambda node: (node, 0.0))\
                       .partitionBy(4)\
                       .cache()

    # 5. Iterative updates
    for i in range(num_iters):
        # Calculate contributions: M * r
        contribs = adj_graph.join(ranks).map(lambda val: (val[1][0][0], val[1][0][1] * val[1][1]))

        # Sum contributions for each destination node
        summed_contribs = contribs.reduceByKey(lambda x, y: x + y)

        # Apply teleport probability and update ranks
        ranks = all_nodes_base.leftOuterJoin(summed_contribs).map(
            lambda val: (val[0], teleport_val + beta * (val[1][1] or 0.0))
        ).partitionBy(4)

        # --- THE CRITICAL FIX ---
        ranks.cache() # Store this iteration's result in memory
        ranks.count() # Action: Forces Spark to evaluate the DAG right now, breaking the lineage build-up

    # Sort results
    final_ranks = ranks.collect()
    final_ranks.sort(key=lambda x: x[1], reverse=True)
    return final_ranks

In [17]:
graph_dir = "/content/drive/MyDrive/Datasets/MLBD/Assignment4/Q3- Graph"
if __name__ == "__main__":
    graph_dir = os.path.join(BASE_PATH, "Q3- Graph")
    small_graph = os.path.join(graph_dir, "small.txt")
    whole_graph = os.path.join(graph_dir, "whole.txt")

    print("--- Running on small.txt ---")
    small_res = run_pagerank(small_graph, num_nodes=100)
    print("Top 5:", small_res[:5])
    print("Bottom 5:", small_res[-5:])

    print("\n--- Running on whole.txt ---")
    whole_res = run_pagerank(whole_graph, num_nodes=1000)

    top_5_ids = [node for node, score in whole_res[:5]]
    bottom_5_ids = [node for node, score in whole_res[-5:]]

    print("Top 5 Node IDs:", top_5_ids)
    print("Bottom 5 Node IDs:", bottom_5_ids)

    sc.stop()

--- Running on small.txt ---
Top 5: [(53, 0.0357312022326716), (14, 0.03417090697259136), (40, 0.03363008718974388), (1, 0.03000597947978861), (27, 0.029720144201405382)]
Bottom 5: [(89, 0.003922466019802268), (37, 0.003808204291611451), (81, 0.003695351749360991), (59, 0.003669860660127284), (85, 0.003409694077402821)]

--- Running on whole.txt ---
Top 5 Node IDs: [263, 537, 965, 243, 285]
Bottom 5 Node IDs: [408, 424, 62, 93, 558]
